In [1]:
!pip install pandas tqdm spacy pyarrow ocrmypdf
!python -m spacy download fr_core_news_sm


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


     ---------------------------------------- 0.0/16.3 MB ? eta -:--:--
     --- ------------------------------------ 1.3/16.3 MB 10.4 MB/s eta 0:00:02
     ---------------- ----------------------- 6.8/16.3 MB 20.6 MB/s eta 0:00:01
     ---------------------------------------- 16.3/16.3 MB 31.4 MB/s  0:00:00
[+] Download and installation successful
You can now load the package via spacy.load('fr_core_news_sm')



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import re
import subprocess
import tempfile
from pathlib import Path
from typing import Optional, Dict, Any, List

import pandas as pd
from tqdm import tqdm
import spacy


# =========================
# LOAD SPACY MODEL
# =========================

try:
    nlp = spacy.load("fr_core_news_sm", disable=["ner"])
except OSError:
    raise RuntimeError(
        "Le modèle spaCy 'fr_core_news_sm' n'est pas installé.\n"
        "Exécutez : python -m spacy download fr_core_news_sm"
    )

if "parser" not in nlp.pipe_names and "senter" not in nlp.pipe_names:
    nlp.add_pipe("sentencizer")


# =========================
# PATTERNS
# =========================

PART_PATTERNS = [
    re.compile(r"^\s*(Première|Deuxième|Troisième|Quatrième|Cinquième)\s+partie\s*$", re.IGNORECASE),
    re.compile(r"^\s*Partie\s+\w+\s*$", re.IGNORECASE),
]

CHAPTER_PATTERNS = [
    re.compile(r"^\s*chapitre\s+(\d+)\s*$", re.IGNORECASE),
    re.compile(r"^\s*(\d+)\s*$"),
]

PRINTED_PAGE_PATTERN = re.compile(r"^\s*(\d{1,4})\s*$")

# détecte des lignes comme :
# La Ballade de Pern - Tome 1 L’AUBE DES DRAGONS
# La Ballade de Pern – Tome 12 ...
# Tome 3 ...
TOME_PATTERNS = [
    re.compile(
        r"^\s*(La\s+Ballade\s+de\s+Pern\s*[-–—]\s*)?Tome\s+(\d{1,2})\s*[:\-–—]?\s*(.+?)\s*$",
        re.IGNORECASE
    ),
    re.compile(
        r"^\s*(La\s+Ballade\s+de\s+Pern\s*[-–—]\s*)?(TOME\s+\d{1,2})\s+(.+?)\s*$",
        re.IGNORECASE
    ),
]


# =========================
# HELPERS
# =========================

def normalize_line(s: str) -> str:
    s = s.replace("\u00ad", "")
    s = s.replace("\ufeff", "")
    s = s.replace("\u200b", "")
    s = s.replace("’", "'")
    s = re.sub(r"[ \t]+", " ", s)
    return s.strip()


def looks_like_header_footer(line: str) -> bool:
    if not line:
        return True
    if PRINTED_PAGE_PATTERN.fullmatch(line) and len(line) <= 4:
        return True
    return False


def update_structure(line: str, current: Dict[str, Optional[str]]) -> None:
    for pat in PART_PATTERNS:
        if pat.match(line):
            current["part"] = line
            current["chapter"] = None
            return

    for pat in CHAPTER_PATTERNS:
        m = pat.match(line)
        if m:
            chap = m.group(1) if m.groups() else line
            current["chapter"] = f"Chapter {chap}"
            return


def extract_tome_title(line: str) -> Optional[str]:
    """
    Return a normalized tome title if the line looks like a tome heading.
    Example output:
    La Ballade de Pern - Tome 1 L'AUBE DES DRAGONS
    """
    l = normalize_line(line)

    for pat in TOME_PATTERNS:
        m = pat.match(l)
        if not m:
            continue

        groups = m.groups()

        # pattern 1: prefix, number, title
        if len(groups) == 3 and groups[1].isdigit():
            tome_num = groups[1]
            tome_title = groups[2].strip(" -–—:")
            return f"La Ballade de Pern - Tome {tome_num} {tome_title}"

        # pattern 2: prefix, "TOME X", title
        if len(groups) == 3:
            tome_label = groups[1].strip()
            tome_title = groups[2].strip(" -–—:")
            return f"La Ballade de Pern - {tome_label.title()} {tome_title}"

    return None


# =========================
# PDF -> TEXT PER PAGE
# =========================

def pdf_to_pages(pdf_path: Path) -> List[str]:
    if not pdf_path.exists():
        raise FileNotFoundError(f"PDF not found: {pdf_path.resolve()}")

    with tempfile.TemporaryDirectory() as td:
        out_txt = Path(td) / "out.txt"

        result = subprocess.run(
            ["pdftotext", "-layout", str(pdf_path), str(out_txt)],
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True
        )

        if result.returncode != 0:
            raise RuntimeError(
                f"pdftotext failed for {pdf_path}\n"
                f"Return code: {result.returncode}\n"
                f"stderr:\n{result.stderr}"
            )

        full_text = out_txt.read_text(encoding="utf-8", errors="replace")

    pages = full_text.split("\f")
    if pages and not pages[-1].strip():
        pages = pages[:-1]
    return pages


# =========================
# REFLOW WITH PAGE PROVENANCE
# =========================

SENTENCE_END_RE = re.compile(r"[.!?…]+[\"”’»)]?\s*$")
DIALOGUE_DASH_RE = re.compile(r"^\s*[-–—]\s*")
HARD_BREAK_RE = re.compile(r"^\s*$")


def should_join_with_space(prev: str, cur: str) -> bool:
    if not prev:
        return False
    if SENTENCE_END_RE.search(prev):
        return False
    if DIALOGUE_DASH_RE.match(cur):
        return False
    return True


def reflow_pages_with_provenance(
    page_texts: List[str],
    drop_headers_footers: bool = True,
    drop_empty: bool = True,
) -> List[Dict[str, Any]]:
    chunks: List[Dict[str, Any]] = []

    current_text = ""
    current_start_page = 1
    current_end_page = 1
    prev_line = ""

    for i, page_text in enumerate(page_texts):
        pdf_page = i + 1
        raw_lines = page_text.splitlines()

        lines = [normalize_line(x) for x in raw_lines]

        if drop_headers_footers:
            lines = [x for x in lines if not looks_like_header_footer(x)]

        if drop_empty:
            lines = [x for x in lines if x]

        for l in lines:
            if not l:
                continue

            if not current_text:
                current_text = l
                current_start_page = pdf_page
                current_end_page = pdf_page
                prev_line = l
                continue

            # hyphenation across line break
            if prev_line.endswith("-") and not prev_line.endswith(" -"):
                current_text = current_text[:-1] + l
            elif should_join_with_space(prev_line, l):
                current_text += " " + l
            else:
                current_text += "\n" + l

            prev_line = l
            current_end_page = pdf_page

        if current_text.strip() and SENTENCE_END_RE.search(current_text.splitlines()[-1]):
            chunks.append({
                "text": current_text.strip(),
                "start_pdf_page": current_start_page,
                "end_pdf_page": current_end_page
            })
            current_text = ""
            prev_line = ""
            current_start_page = pdf_page + 1
            current_end_page = pdf_page + 1

    if current_text.strip():
        chunks.append({
            "text": current_text.strip(),
            "start_pdf_page": current_start_page,
            "end_pdf_page": current_end_page
        })

    return chunks


# =========================
# BUILD SENTENCE TABLE
# =========================

def build_sentences_table(
    page_texts: List[str],
    default_book_title: str = "La Ballade de Pern",
    keep_headers_footers: bool = False,
    keep_empty: bool = False,
) -> pd.DataFrame:
    current = {
        "part": None,
        "chapter": None,
        "book": default_book_title,
    }

    page_struct: Dict[int, Dict[str, Optional[str]]] = {}

    # Scan each page in order to detect tome / part / chapter metadata
    for i, ptxt in enumerate(page_texts):
        pdf_page = i + 1
        raw_lines = ptxt.splitlines()
        norm_lines = [normalize_line(x) for x in raw_lines]

        for l in norm_lines:
            if not l:
                continue

            tome_title = extract_tome_title(l)
            if tome_title:
                current["book"] = tome_title

            update_structure(l, current)

        page_struct[pdf_page] = {
            "book": current["book"],
            "part": current["part"],
            "chapter": current["chapter"],
        }

    chunks = reflow_pages_with_provenance(
        page_texts=page_texts,
        drop_headers_footers=not keep_headers_footers,
        drop_empty=not keep_empty,
    )

    rows: List[Dict[str, Any]] = []
    sent_id = 0

    for ch in tqdm(chunks, desc="Sentence splitting"):
        text = ch["text"]
        start_p = int(ch["start_pdf_page"])
        end_p = int(ch["end_pdf_page"])

        meta = page_struct.get(start_p, {})
        book = meta.get("book", default_book_title)
        part = meta.get("part")
        chapter = meta.get("chapter")

        doc = nlp(text.replace("\n", " "))

        for s in doc.sents:
            sent = s.text.strip()
            if not sent:
                continue

            sent_id += 1
            rows.append(
                {
                    "sentence_id": sent_id,
                    "book": book,               # <- tome title stored here
                    "part": part,
                    "chapter": chapter,
                    "pdf_page": start_p,        # <- single page column
                    "start_pdf_page": start_p,
                    "end_pdf_page": end_p,
                    "text": sent,
                }
            )

    return pd.DataFrame(rows)


# =========================
# PIPELINE
# =========================

def run_pipeline(
    input_pdf: Path,
    out_dir: Path = Path("outputs"),
    default_book_title: str = "La Ballade de Pern",
    keep_headers_footers: bool = False,
    keep_empty: bool = False,
):
    out_dir.mkdir(parents=True, exist_ok=True)

    page_texts = pdf_to_pages(input_pdf)

    df = build_sentences_table(
        page_texts=page_texts,
        default_book_title=default_book_title,
        keep_headers_footers=keep_headers_footers,
        keep_empty=keep_empty,
    )

    out_csv = out_dir / f"{input_pdf.stem}.sentences_with_pages.csv"
    df.to_csv(out_csv, index=False, encoding="utf-8")

    print(f"Sentences: {len(df):,}")
    print(f"CSV: {out_csv}")

    return df

In [4]:
from pathlib import Path

df = run_pipeline(
    input_pdf=Path("La-ballade-de-Pern-intégrale-McCaffrey.pdf"),
    out_dir=Path("outputs"),
    default_book_title="La Ballade de Pern"
)

df.head()

IndentationError: unexpected indent (701191647.py, line 4)